In [216]:
import numpy as np
import pandas as pd
import ast
import nltk
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
from difflib import get_close_matches
import pickle
from pathlib import Path

In [217]:
movies=pd.read_csv('movies.csv')
credits=pd.read_csv('credits.csv')

In [73]:
movies.shape

(4803, 20)

In [71]:
credits.shape

(4803, 4)

In [72]:
movies.columns.tolist()

['budget',
 'genres',
 'homepage',
 'id',
 'keywords',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'production_companies',
 'production_countries',
 'release_date',
 'revenue',
 'runtime',
 'spoken_languages',
 'status',
 'tagline',
 'title',
 'vote_average',
 'vote_count']

In [74]:
credits.columns.tolist()

['movie_id', 'title', 'cast', 'crew']

In [75]:
movies.isnull().sum()

budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
dtype: int64

In [76]:
credits.isnull().sum()

movie_id    0
title       0
cast        0
crew        0
dtype: int64

In [78]:
movies.duplicated().sum()

np.int64(0)

In [79]:
credits.duplicated().sum()

np.int64(0)

In [80]:
movies['id'].duplicated().sum()

np.int64(0)

In [96]:
credits = credits.drop(columns=['title'])

In [97]:
# merge both dataframes
movies=movies.merge(
    credits,
    left_on='id',
    right_on='movie_id',
    how='inner',
    validate='one_to_one'
)

In [100]:
movies.columns.tolist()

['budget',
 'genres',
 'homepage',
 'id',
 'keywords',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'production_companies',
 'production_countries',
 'release_date',
 'revenue',
 'runtime',
 'spoken_languages',
 'status',
 'tagline',
 'title',
 'vote_average',
 'vote_count',
 'movie_id',
 'cast',
 'crew']

In [ ]:
# feature enginnering
# id
# title
# overview
# keywords
# genres
# production_companies
# cast
# crew

In [231]:
print(movies.columns.tolist())
print(movies_clean.columns.tolist())

['id', 'title', 'tags']
['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'tags']


In [230]:
movies_clean=movies[[
    'id',
    'title',
    'overview',
    'genres',
    'keywords',
    'cast',
    'crew',
]].copy()

KeyError: "['overview', 'genres', 'keywords', 'cast', 'crew', 'poster_path'] not in index"

In [109]:
movies_clean.columns.tolist()

['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']

In [110]:
movies_clean.head()

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [111]:
movies_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        4803 non-null   int64
 1   title     4803 non-null   str  
 2   overview  4800 non-null   str  
 3   genres    4803 non-null   str  
 4   keywords  4803 non-null   str  
 5   cast      4803 non-null   str  
 6   crew      4803 non-null   str  
dtypes: int64(1), str(6)
memory usage: 262.8 KB


In [112]:
movies_clean.iloc[0]

id                                                      19995
title                                                  Avatar
overview    In the 22nd century, a paraplegic Marine is di...
genres      [{"id": 28, "name": "Action"}, {"id": 12, "nam...
keywords    [{"id": 1463, "name": "culture clash"}, {"id":...
cast        [{"cast_id": 242, "character": "Jake Sully", "...
crew        [{"credit_id": "52fe48009251416c750aca23", "de...
Name: 0, dtype: object

In [113]:
movies_clean.isnull().sum()

id          0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [115]:
movies_clean['overview']=movies_clean['overview'].fillna('')

In [116]:
movies_clean.isnull().sum()

id          0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [ ]:
# data transformation

In [131]:
# convert to list function
def convert_to_list(value):

    if value is None or pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        if value.strip() == '':
            return []

        result = ast.literal_eval(value)

        if isinstance(result, list):
            return result

    return []

In [132]:

columns_to_convert=['genres','keywords','cast','crew']
for col in columns_to_convert:
    movies_clean[col]=movies_clean[col].apply(convert_to_list)

In [133]:
movies_clean['genres'][0]

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 878, 'name': 'Science Fiction'}]

In [134]:
# extract name function ---> for genres and keywords
def extract_names(obj):
    names=[]
    for item in obj or []:
        if isinstance(item,dict) and item.get('name'):
            names.append(item['name'])
    return names
        

In [135]:
movies_clean['genres']=movies_clean['genres'].apply(extract_names)
movies_clean['keywords']=movies_clean['keywords'].apply(extract_names)

In [138]:
# extract cast
def extract_cast(obj):
    names=[]
    for item in (obj or [])[:3]:
        if isinstance(item,dict) and item.get('name'):
            names.append(item['name'])
    return names
        

In [139]:
movies_clean['cast']=movies_clean['cast'].apply(extract_cast)

In [142]:
# extract director name from the crew
def extract_director(obj):
    for item in obj or []:
        if isinstance(item,dict) and item.get('job')=='Director':
            return [item['name']]
    return []

In [143]:
movies_clean['crew']=movies_clean['crew'].apply(extract_director)

In [144]:
movies_clean.head()

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


In [149]:
for col in columns_to_convert:
   print(movies_clean[col].apply(lambda x:x is None).sum()) # no none values anymore 

0
0
0
0


In [151]:
# Combine all features into one column(tags)
movies_clean['tags']=(
    movies_clean['overview']+' '+
    movies_clean['genres'].apply(lambda x:' '.join(x))+' '+
    movies_clean['keywords'].apply(lambda x:' '.join(x))+' '+
    movies_clean['cast'].apply(lambda x:' '.join(x))+' '+
    movies_clean['crew'].apply(lambda x:' '.join(x))
)

In [153]:
movies_final=movies_clean[['id','title','tags']].copy()

In [155]:
movies_final.head()

,id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [161]:
# normalize the text before vertorization
ps=PorterStemmer()


In [170]:
# stemming function
def stem_text(text):
    words=[]
    for word in text.split():
        words.append(ps.stem(word))

    return " ".join(words)

In [171]:
print(movies_final['tags'].apply(type).value_counts())

tags
<class 'str'>    4803
Name: count, dtype: int64


In [172]:
bad_rows=movies_final[
    ~movies_final['tags'].apply(lambda x: isinstance(x,str))
]
print(bad_rows[['id','title','tags']])

Empty DataFrame
Columns: [id, title, tags]
Index: []


In [173]:
movies_final['tags'] = movies_final['tags'].fillna('').astype(str)

movies_final['tags'] = movies_final['tags'].str.lower()

movies_final['tags'] = movies_final['tags'].apply(stem_text)

In [189]:
# countVectorizer ---> CountVectorizer creates a vocabulary and counts the occurrence of each word(Bag of Words (BoW)).\

cv=CountVectorizer(
    max_features=5000,
    stop_words='english'
)

In [190]:
# fit and transform 
vectors=cv.fit_transform(movies_final['tags'])

In [193]:
vectors.shape

(4803, 5000)

In [196]:
# cosine_similarity ---> It measures how similar two numerical vectors are based on the angle between them.
similarity=cosine_similarity(vectors,dense_output=False)


In [197]:
similarity.shape

(4803, 4803)

In [198]:
print("Final dataset:", movies_final.shape)

print("Vector dimensions:", vectors.shape)

print("Similarity dimensions:", similarity.shape)

print("\nSample vocabulary:")
print(cv.get_feature_names_out()[:20])

Final dataset: (4803, 3)
Vector dimensions: (4803, 5000)
Similarity dimensions: (4803, 4803)

Sample vocabulary:
['000' '007' '10' '100' '11' '12' '13' '14' '15' '16' '16th' '17' '17th'
 '18' '18th' '19' '1910' '1920' '1930' '1940']


In [209]:
# normalize title
def normalize_title(title):
    return re.sub(r'[^a-z0-9]', '', title.lower())

# recommend movies function
def recommend_movies(movie):

    # Normalize the user's input
    search_title = normalize_title(movie)

    # Normalize all dataset titles
    normalized_titles = movies_final['title'].apply(normalize_title)

    # Find exact normalized match
    matches = movies_final[normalized_titles == search_title]

    if matches.empty:

        # Find suggestions for an incorrect or approximate title
        suggestions = get_close_matches(
            search_title,
            normalized_titles.tolist(),
            n=5,
            cutoff=0.45
        )

        if suggestions:
            print("Movie not found. Did you mean:")

            for suggestion in suggestions:
                original_title = movies_final.loc[
                    normalized_titles == suggestion, 'title'
                ].iloc[0]

                print("-", original_title)
        else:
            print("Sorry! No similar movie title found.")

        return

    # Get the index of the matched movie
    movie_index = matches.index[0]

    # Get similarity scores
    distances = similarity[movie_index]

    if hasattr(distances, 'toarray'):
        distances = distances.toarray().flatten()
    else:
        distances = np.asarray(distances).flatten()

    # Sort by similarity and exclude the selected movie
    movie_list = sorted(
        enumerate(distances),
        key=lambda x: x[1],
        reverse=True
    )[1:6]

    print(f"\nMovies similar to {movies_final.iloc[movie_index]['title']}:\n")

    for i, score in movie_list:
        print(
            movies_final.iloc[i]['title'],
            "-",
            round(float(score), 3)
        )

In [211]:
recommend_movies('xyzrandommovie')

Movie not found. Did you mean:
- Extreme Movie
- Hey Arnold! The Movie
- Random Hearts
- Date Movie
- Doug's 1st Movie


In [213]:
 # Save movie data
with open('movies.pkl', 'wb') as file:
    pickle.dump(movies_final, file)

# Save similarity matrix
with open('similarity.pkl', 'wb') as file:
    pickle.dump(similarity, file)

print("Model files saved successfully!")

Model files saved successfully!


In [214]:
with open('movies.pkl', 'rb') as file:
    movies_data = pickle.load(file)

with open('similarity.pkl', 'rb') as file:
    similarity_data = pickle.load(file)

print("Movies:", movies_data.shape)
print("Similarity:", similarity_data.shape)

Movies: (4803, 3)
Similarity: (4803, 4803)


In [218]:
file_path = Path('movies.pkl')

print("File exists:", file_path.exists())
print("File size:", file_path.stat().st_size, "bytes")

File exists: True
File size: 2230670 bytes


In [220]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.model_utils import save_artifacts, load_artifacts

In [221]:
save_artifacts(movies_final, similarity)

Model artifacts saved successfully!


In [222]:
movies_data, similarity_data = load_artifacts()

print("Movies:", movies_data.shape)
print("Similarity:", similarity_data.shape)

Movies: (4803, 3)
Similarity: (4803, 4803)


In [226]:
from src.recommendation import recommend_movies

recommendations = recommend_movies("Avatar")

for movie in recommendations:
    print(movie)

{'id': np.int64(679), 'title': 'Aliens', 'similarity_score': 0.49}
{'id': np.int64(811), 'title': 'Silent Running', 'similarity_score': 0.411}
{'id': np.int64(348), 'title': 'Alien', 'similarity_score': 0.398}
{'id': np.int64(698), 'title': 'Moonraker', 'similarity_score': 0.388}
{'id': np.int64(2067), 'title': 'Mission to Mars', 'similarity_score': 0.38}


In [227]:
# Exact movie title
recommend_movies("Batman Begins")

# Different capitalization
recommend_movies("AVATAR")

# Approximate title
recommend_movies("Avatr")

# Invalid title
recommend_movies("xyzrandommovie")

ValueError: Movie not found. Suggestions: ['Avatar', 'The Aviator', 'Salvador', 'Water', 'After']

In [229]:
# Test 1: Exact match
print(recommend_movies("Avatar"))

# Test 2: Different capitalization
print(recommend_movies("AVATAR"))

# Test 3: Misspelled title
try:
    recommend_movies("Avatr")
except ValueError as error:
    print(error)

# Test 4: Completely invalid title
try:
    recommend_movies("xyzrandommovie")
except ValueError as error:
    print(error)

[{'id': np.int64(679), 'title': 'Aliens', 'similarity_score': 0.49}, {'id': np.int64(811), 'title': 'Silent Running', 'similarity_score': 0.411}, {'id': np.int64(348), 'title': 'Alien', 'similarity_score': 0.398}, {'id': np.int64(698), 'title': 'Moonraker', 'similarity_score': 0.388}, {'id': np.int64(2067), 'title': 'Mission to Mars', 'similarity_score': 0.38}]
[{'id': np.int64(679), 'title': 'Aliens', 'similarity_score': 0.49}, {'id': np.int64(811), 'title': 'Silent Running', 'similarity_score': 0.411}, {'id': np.int64(348), 'title': 'Alien', 'similarity_score': 0.398}, {'id': np.int64(698), 'title': 'Moonraker', 'similarity_score': 0.388}, {'id': np.int64(2067), 'title': 'Mission to Mars', 'similarity_score': 0.38}]
Movie not found. Suggestions: ['Avatar', 'The Aviator', 'Salvador', 'Water', 'After']
Movie not found. Suggestions: ['Extreme Movie', 'Hey Arnold! The Movie', 'Random Hearts', 'Date Movie', "Doug's 1st Movie"]
